In [117]:
# A nested hyperparameter search for the two-branch model
# ===========================================================================
# Section III-E of the manuscript records a limitation: the Optuna study that
# fixed T_w, L, d and the learning rate was run on the SINGLE-STREAM model that
# preceded the published architecture, and its objective was computed on a
# validation loader built from the genuine cross-domain session that the paper
# also reports on. Neither the architecture nor the protocol of that search
# matches what the paper now proposes.
#
# This notebook re-runs the search against the published two-branch model, with
# the reported test session withheld from the search entirely.
#
# What is and is not independent here
# -----------------------------------
# INDEPENDENT.  The genuine user's laptop session -- the cross-domain test set
#   of every number in the paper -- is removed from the roster before any trial
#   runs. The search neither trains on it nor scores against it. The objective
#   is a desktop-to-desktop hold-out instead. This is the flaw Section III-E
#   admits, and it is the one this notebook fixes.
#
# NOT INDEPENDENT.  The impostors used to score trials (INNER_HELDOUTS) also
#   appear as outer folds in Table 1, because ten participants cannot support
#   disjoint inner and outer impostor pools without shrinking the reported
#   rotation below nine folds. A configuration that happens to suit those three
#   impostors could therefore flatter three of the nine outer folds. Report the
#   search as partially nested; do not call it nested.
#
# Output: results/optuna-nested/optuna_trials.csv and best_params.json.
# Nothing here overwrites the published configuration -- adopting the result is
# a separate decision, made after comparing it against the values in
# FIXED_PARAMS.

In [ ]:
# Part 1: Imports, device selection, and experiment configuration
# ---------------------------------------------------------------------------
# This notebook supersedes the single-session version. Every recording folder
# found under DATA_ROOT/<participant>/ is used, and the train / validation /
# test partitions are kept disjoint (see PROTOCOL below).

import os
import copy
import json
import itertools
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import roc_curve, auc

warnings.filterwarnings('ignore')

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

# ---------------------------------------------------------------------------
# Data location and participant roster
# ---------------------------------------------------------------------------
# --- data location ----------------------------------------------------------
# Resolved against a candidate list rather than fixed to the working directory.
# These notebooks moved one level up when the superseded ones were collected
# into old/, and every cwd-relative path broke silently at that moment: the
# roster override below was left behind in old/, so the anonymized P01-P10 names
# were used against directories named for the real ids, and session discovery
# returned nothing. Searching a few obvious places and printing which one was
# taken costs nothing and makes that failure impossible to have without noticing.
#
# Set DATA_ROOT_OVERRIDE to an absolute path to skip the search entirely.
DATA_ROOT_OVERRIDE = None

_DATA_CANDIDATES = ['./data', '../data', './old/data', '../old/data', '../../data']

def _resolve_dir(candidates, override=None):
    if override:
        return override if os.path.isdir(override) else None
    for c in candidates:
        if os.path.isdir(c):
            return c
    return None

DATA_ROOT = _resolve_dir(_DATA_CANDIDATES, DATA_ROOT_OVERRIDE) or './data'
print(f"DATA_ROOT   : {DATA_ROOT!r} -> {os.path.abspath(DATA_ROOT)}"
      f"{'' if os.path.isdir(DATA_ROOT) else '   [NOT FOUND]'}")
PARTICIPANTS = ['P01', 'P02', 'P03', 'P04', 'P05',
                'P06', 'P07', 'P08', 'P09', 'P10']

# Recording folders captured on the LAPTOP keyboard. These form the
# cross-domain test condition for whichever participant is the genuine user.
# Everything not listed here is treated as desktop (training-domain) data.
# Extend this dict as further laptop sessions are recorded.
LAPTOP_FOLDERS = {
    'P01': ['202602041210'],
}

# ---------------------------------------------------------------------------
# Local, non-anonymized override
# ---------------------------------------------------------------------------
# participants_local.json holds the real on-disk folder names, so the notebooks
# run against the collected data without renaming anything. It is gitignored:
# the ids are student numbers and are directly identifying. When the file is
# absent -- the state the public repository should always be in -- the
# anonymized P01-P10 roster above is used instead.
#
# Every table, CSV and figure below is written using ANON_LABEL, so output
# produced from the real data is already anonymized and can go straight into
# the manuscript.
# Searched beside the notebook, beside the data, and in old/ -- the three places
# it has actually lived. If it exists anywhere and is not picked up, the run
# proceeds under the wrong roster and finds nothing, which is the error this
# resolution exists to prevent.
_LOCAL_CANDIDATES = [
    'participants_local.json',
    '../participants_local.json',
    'old/participants_local.json',
    '../old/participants_local.json',
    os.path.join(os.path.dirname(DATA_ROOT), 'participants_local.json'),
]
LOCAL_CONFIG = next((c for c in _LOCAL_CANDIDATES if os.path.exists(c)),
                    'participants_local.json')
ANON_LABEL = {}
DEFAULT_GENUINE = 'P01'
DEFAULT_HELDOUT = 'P10'

if os.path.exists(LOCAL_CONFIG):
    with open(LOCAL_CONFIG, encoding='utf-8') as _f:
        _cfg = json.load(_f)
    PARTICIPANTS = list(_cfg['participants'])
    LAPTOP_FOLDERS = dict(_cfg.get('laptop_folders', {}))
    ANON_LABEL = dict(_cfg.get('labels', {}))
    DEFAULT_GENUINE = _cfg.get('default_genuine', PARTICIPANTS[0])
    DEFAULT_HELDOUT = _cfg.get('default_heldout_impostor', PARTICIPANTS[-1])
    print(f"Loaded {LOCAL_CONFIG}: {len(PARTICIPANTS)} participants "
          f"(output will be labelled {ANON_LABEL.get(PARTICIPANTS[0], '?')}"
          f"-{ANON_LABEL.get(PARTICIPANTS[-1], '?')})")
else:
    # Not an error on its own: the public repository is meant to be in this
    # state. It is only wrong when the data directories are named for the real
    # ids, and that combination produces an empty discovery in Part 3 with no
    # other symptom, so name the possibility here rather than there.
    print(f"participants_local.json not found in {_LOCAL_CANDIDATES}")
    print(f"  (cwd {os.getcwd()})")
    print( "  -> using the anonymized P01-P10 roster. If the directories under")
    print( "     DATA_ROOT are named for the real participant ids, discovery will")
    print( "     find nothing; put participants_local.json in one of the paths")
    print( "     above, or point DATA_ROOT_OVERRIDE at the right directory.")


def label(pid):
    """Anonymized display name for a participant id."""
    return ANON_LABEL.get(pid, pid)

# ---------------------------------------------------------------------------
# Protocol switches
# ---------------------------------------------------------------------------
# STRICT_PROTOCOL = True  -> early stopping uses a validation split drawn only
#                            from the training domain; the reported EER is
#                            computed on a test set the model never saw.
# STRICT_PROTOCOL = False -> reproduces the original notebooks, where the same
#                            loader served as both the early-stopping validation
#                            set and the reported evaluation set (and included
#                            the held-out impostor). Kept only for comparison.
STRICT_PROTOCOL = True

# The manuscript states that the first repetition of each recording is dropped
# for device habituation. The original notebooks applied this to sentence '1'
# only, and that is kept as the default here: in the collected data the other
# sentences are a single take whose session_count never exceeds 1, so filtering
# them on session_count > 1 discards those recordings entirely and silently.
# Set to None to apply the filter to every sentence -- but check the data-yield
# table printed by preflight() first, and reconcile the wording in the
# manuscript with whichever setting is used.
FIRST_REPETITION_SENTENCES = {'1'}

TRAIN_RATIO = 0.8          # chronological split inside each recording unit
SENTENCE_IDS = ['1', '2', '3']

ALL_SENSOR_COLS = [
    'R_S_1', 'R_S_2', 'R_S_3', 'R_S_4', 'R_S_5',
    'R_S_6', 'R_S_7', 'R_S_8', 'R_S_9', 'R_S_10',
    'R_Gyro_X', 'R_Gyro_Y', 'R_Gyro_Z',
    'R_Accel_X', 'R_Accel_Y', 'R_Accel_Z',
    'R_Mag_X', 'R_Mag_Y', 'R_Mag_Z',
]
N_CHANNELS = len(ALL_SENSOR_COLS)
print(f"Sensor channels: {N_CHANNELS}")


In [119]:
# Part 2: Data layer
# ---------------------------------------------------------------------------
# Session discovery, per-unit loading, windowing, splitting, group-aware
# smoothing and metrics. A "unit" is one (participant, folder, sentence)
# recording. Building features per unit guarantees that no feature window ever
# spans two recordings and gives each test segment a natural smoothing group.

def discover_sessions(root_dir, pid):
    """Every recording folder for `pid` holding at least one usable sentence.
    Folder names are yyyymmddHHMM stamps, so sorting is chronological."""
    pdir = os.path.join(root_dir, pid)
    if not os.path.isdir(pdir):
        return []
    found = []
    for name in sorted(os.listdir(pdir)):
        fdir = os.path.join(pdir, name)
        if not os.path.isdir(fdir):
            continue
        for s_id in SENTENCE_IDS:
            g = os.path.join(fdir, s_id, f"{pid}_{s_id}_glove_R.csv")
            k = os.path.join(fdir, s_id, f"{pid}_{s_id}_kb.csv")
            if os.path.exists(g) and os.path.exists(k):
                found.append(name)
                break
    return found


def load_unit(root_dir, pid, folder, s_id, drop_first_repetition=True):
    sdir = os.path.join(root_dir, pid, folder, s_id)
    gpath = os.path.join(sdir, f"{pid}_{s_id}_glove_R.csv")
    kpath = os.path.join(sdir, f"{pid}_{s_id}_kb.csv")
    if not (os.path.exists(gpath) and os.path.exists(kpath)):
        return None, None
    try:
        g_df = pd.read_csv(gpath)
        k_df = pd.read_csv(kpath)
    except Exception:
        return None, None
    if drop_first_repetition:
        if 'session_count' in g_df.columns:
            g_df = g_df[g_df['session_count'] > 1]
        if 'session_count' in k_df.columns:
            k_df = k_df[k_df['session_count'] > 1]
    if g_df.empty or k_df.empty:
        return None, None
    return g_df.reset_index(drop=True), k_df.reset_index(drop=True)


def clip_pressure_outliers(glove_df, threshold=1000.0):
    """Clip nonphysical readings to the largest valid value in this unit.
    Returns a copy: the original notebooks mutated the caller's DataFrame, so
    clipped values leaked across window-length settings and across folds."""
    out = glove_df.copy()
    for col in [c for c in ALL_SENSOR_COLS if c.startswith('R_S_')]:
        if col not in out.columns:
            continue
        vals = out[col].values.astype(np.float64)
        valid = vals < threshold
        if np.any(valid):
            out.loc[out[col] >= threshold, col] = float(np.max(vals[valid]))
    return out


def _interpolate(features, target_len):
    x = np.linspace(0, 1, features.shape[0])
    x_new = np.linspace(0, 1, target_len)
    return interp1d(x, features, axis=0, kind='linear',
                    fill_value="extrapolate")(x_new)


def build_windows(glove_df, kb_df, window_ms, target_len, min_samples=5):
    """One normalized window per keystroke -> (N, target_len, 19).
    Z-score normalization is per window and per channel (over time)."""
    g = clip_pressure_outliers(glove_df)
    order = np.argsort(g['glove_timestamp'].values, kind='stable')
    glove_ts = g['glove_timestamp'].values[order]
    glove_vals = g[ALL_SENSOR_COLS].values.astype(np.float32)[order]
    kb_ts = np.sort(kb_df['kb_timestamp_pressed'].values)

    half = window_ms / 2000.0
    out = []
    for ts in kb_ts:
        i0 = np.searchsorted(glove_ts, ts - half, side='left')
        i1 = np.searchsorted(glove_ts, ts + half, side='right')
        seg = glove_vals[i0:i1]
        if len(seg) < min_samples:
            continue
        out.append(_interpolate(seg, target_len))

    if not out:
        return np.empty((0, target_len, N_CHANNELS), dtype=np.float32)
    arr = np.asarray(out, dtype=np.float32)
    mean = arr.mean(axis=1, keepdims=True)
    std = arr.std(axis=1, keepdims=True)
    std[std < 1e-6] = 1e-6
    return (arr - mean) / std


def chronological_split(n, train_ratio=TRAIN_RATIO):
    """Time-preserving index split, applied *inside* one unit.
    This replaces chronological_split_per_text(), which cut the concatenated
    array into three equal blocks -- an assumption that silently breaks as soon
    as more than one recording folder is loaded."""
    n_train = int(n * train_ratio)
    return np.arange(0, n_train), np.arange(n_train, n)


def chronological_split3(n, ratios=(0.6, 0.2, 0.2)):
    """Time-preserving 3-way index split inside one recording unit.
    Used only by the hyperparameter search, which needs its own held-out
    portion of the *training domain* so that the real test set stays untouched."""
    a = int(n * ratios[0])
    b = a + int(n * ratios[1])
    return np.arange(0, a), np.arange(a, b), np.arange(b, n)


def smooth_within_group(scores, window_size):
    scores = np.asarray(scores, dtype=np.float64)
    if window_size <= 1:
        return scores
    if len(scores) < window_size:
        return np.empty((0,), dtype=np.float64)
    csum = np.cumsum(np.insert(scores, 0, 0.0))
    return (csum[window_size:] - csum[:-window_size]) / float(window_size)


def smooth_grouped(scores, labels, groups, window_size):
    """Moving average applied separately within each recording unit.
    The original reset its buffer only when the *label* changed, so scores from
    consecutive impostors (both label 0) were averaged across subject
    boundaries, manufacturing samples that belong to no one."""
    scores, labels = np.asarray(scores), np.asarray(labels)
    # Build a flat 1-D object array explicitly: np.asarray() on a list of
    # tuples would produce a 2-D array and make the comparisons element-wise.
    g = np.empty(len(groups), dtype=object)
    for i, v in enumerate(groups):
        g[i] = v
    groups = g
    out_s, out_l, start = [], [], 0
    for i in range(1, len(groups) + 1):
        if i == len(groups) or groups[i] != groups[start]:
            sm = smooth_within_group(scores[start:i], window_size)
            if len(sm):
                out_s.append(sm)
                out_l.append(np.full(len(sm), labels[start]))
            start = i
    if not out_s:
        return np.empty((0,)), np.empty((0,))
    return np.concatenate(out_s), np.concatenate(out_l)


def eer_and_metrics(y_true, y_score, target_far=0.001):
    """EER as the mean of FPR and FNR at the crossing point. The original
    reported FPR alone, which is biased low on a coarse ROC grid."""
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    nan = dict(eer=np.nan, auc=np.nan, frr_at_far=np.nan, threshold=np.nan)
    # A diverged model produces non-finite scores; report them as undefined
    # rather than letting roc_curve raise several frames away from the cause.
    if len(y_score) == 0 or not np.all(np.isfinite(y_score)):
        return nan
    if len(np.unique(y_true)) < 2:
        return nan
    fpr, tpr, thr = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fnr - fpr)))
    try:
        frr = float(1.0 - interp1d(fpr, tpr, kind='linear')(target_far))
    except Exception:
        cand = np.where(fpr >= target_far)[0]
        frr = float(1.0 - tpr[cand[0]]) if len(cand) else 1.0
    return dict(eer=float((fpr[idx] + fnr[idx]) / 2.0),
                auc=float(auc(fpr, tpr)),
                frr_at_far=float(max(0.0, min(1.0, frr))),
                threshold=float(thr[idx]))


def smoothing_table(probs, labels, groups, windows):
    """EER / AUC / FRR@FAR=0.1% for an arbitrary set of smoothing windows,
    computed from stored per-keystroke scores. No retraining required."""
    rows = []
    for w in windows:
        sc, lb = smooth_grouped(probs, labels, groups, w)
        m = (eer_and_metrics(lb, sc) if len(lb)
             else dict(eer=np.nan, auc=np.nan, frr_at_far=np.nan))
        rows.append({'W': w, 'n_scores': len(lb), **m})
    return pd.DataFrame(rows)


class BoundedFoldCache(dict):
    """LRU-bounded cache of built folds.

    Each fold holds roughly 75 MB of float32 features on this dataset, and the
    key space is heldout x sentence x window_ms x target_len. Caching without a
    bound would reach several GB partway through a search.
    """
    def __init__(self, maxsize=6):
        super().__init__()
        self.maxsize = maxsize
        self._order = []

    def get_or_build(self, key, build):
        if key in self:
            self._order.remove(key)
            self._order.append(key)
            return self[key]
        val = build()
        self[key] = val
        self._order.append(key)
        while len(self._order) > self.maxsize:
            del self[self._order.pop(0)]
        return val


def separation_margin(gen_scores, imp_scores, q_gen=0.01, q_imp=0.99):
    """Gap between the genuine lower tail and the impostor upper tail.

    Retains resolution when the two classes are perfectly separated and EER is
    exactly 0, which is where EER stops being able to rank anything.
    """
    gen_scores = np.asarray(gen_scores, float)
    imp_scores = np.asarray(imp_scores, float)
    if len(gen_scores) == 0 or len(imp_scores) == 0:
        return float('nan')
    return float(np.quantile(gen_scores, q_gen) - np.quantile(imp_scores, q_imp))


TIE_BREAK_WEIGHT = 1e-4   # must stay below one test sample's worth of EER

def tuning_objective(eer, margin, weight=TIE_BREAK_WEIGHT):
    """EER, with a margin term that only decides exact ties.

    An inner fold can separate perfectly, in which case every configuration
    scores 0.000% and the search has nothing to optimise. The added term is
    bounded by `weight`, which is smaller than the EER step of a single test
    sample, so a measurable EER difference always outranks it.
    """
    if not np.isfinite(eer):
        return float('inf')
    m = 0.0 if not np.isfinite(margin) else float(np.clip(margin, -1.0, 1.0))
    return float(eer) + weight * (1.0 - m) / 2.0


def mean_sd(values, ddof=1):
    """Sample standard deviation. The original used np.std's default ddof=0,
    which understates the spread at n=5 by about 12%."""
    v = np.asarray([x for x in values if x is not None and not np.isnan(x)], float)
    if len(v) == 0:
        return float('nan'), float('nan')
    if len(v) == 1:
        return float(v[0]), 0.0
    return float(v.mean()), float(v.std(ddof=ddof))


In [120]:
# Part 3: Discover every session and print the inventory
# ---------------------------------------------------------------------------
# Check this table before running anything: it is the ground truth for how many
# sessions actually entered the experiment, and the figure that belongs in the
# manuscript's dataset description.

def diagnose_data_root(max_show=8):
    """Explain why discovery found nothing.

    Expected layout:
        DATA_ROOT/<participant>/<yyyymmddHHMM>/<sentence 1|2|3>/
            <participant>_<sentence>_glove_R.csv
            <participant>_<sentence>_kb.csv
    """
    print("-" * 70)
    print("DATA LAYOUT DIAGNOSTIC")
    print("-" * 70)
    print(f"DATA_ROOT           : {DATA_ROOT!r}")
    print(f"resolves to         : {os.path.abspath(DATA_ROOT)}")
    print(f"notebook cwd        : {os.getcwd()}")
    print(f"exists              : {os.path.isdir(DATA_ROOT)}")
    if not os.path.isdir(DATA_ROOT):
        here = sorted(d for d in os.listdir('.') if os.path.isdir(d))[:max_show]
        print(f"\nDirectories next to the notebook: {here}")
        print("\n-> Point DATA_ROOT at the folder that contains the per-participant")
        print("   directories, e.g. DATA_ROOT = '/absolute/path/to/data'.")
        return

    entries = sorted(os.listdir(DATA_ROOT))
    subdirs = [d for d in entries if os.path.isdir(os.path.join(DATA_ROOT, d))]
    print(f"\nSubdirectories ({len(subdirs)}): {subdirs[:max_show]}"
          f"{' ...' if len(subdirs) > max_show else ''}")
    print(f"Roster expects      : {PARTICIPANTS[:max_show]}"
          f"{' ...' if len(PARTICIPANTS) > max_show else ''}")
    matched = [p for p in PARTICIPANTS if p in subdirs]
    print(f"Roster names present: {matched if matched else 'NONE'}")
    if not matched and subdirs:
        print("\n-> The roster does not match the directory names on disk.")
        print("   Either edit PARTICIPANTS, or put the real ids in")
        print(f"   {LOCAL_CONFIG} (searched at {os.path.abspath(LOCAL_CONFIG)}).")

    probe = (matched or subdirs)[:1]
    for pid in probe:
        pdir = os.path.join(DATA_ROOT, pid)
        folders = sorted(d for d in os.listdir(pdir)
                         if os.path.isdir(os.path.join(pdir, d)))
        print(f"\n{pid}/ contains {len(folders)} folders: {folders[:max_show]}")
        for folder in folders[:1]:
            fdir = os.path.join(pdir, folder)
            inner = sorted(os.listdir(fdir))
            print(f"  {folder}/ -> {inner[:max_show]}   (expected {SENTENCE_IDS})")
            for s_id in inner[:1]:
                sdir = os.path.join(fdir, s_id)
                if os.path.isdir(sdir):
                    files = sorted(os.listdir(sdir))
                    print(f"    {s_id}/ -> {files[:max_show]}")
                    print(f"    expected: {pid}_{s_id}_glove_R.csv  and  {pid}_{s_id}_kb.csv")
    print("-" * 70)


SESSIONS = {pid: discover_sessions(DATA_ROOT, pid) for pid in PARTICIPANTS}

if not any(SESSIONS.values()):
    diagnose_data_root()
    raise FileNotFoundError(
        "No recording sessions were discovered under DATA_ROOT. See the "
        "diagnostic above; fix DATA_ROOT (or the participant roster) and rerun "
        "this cell before continuing.")

rows = []
for pid in PARTICIPANTS:
    laptop = [f for f in SESSIONS[pid] if f in LAPTOP_FOLDERS.get(pid, [])]
    desktop = [f for f in SESSIONS[pid] if f not in laptop]
    rows.append({'participant': label(pid),
                 'n_folders': len(SESSIONS[pid]),
                 'n_desktop': len(desktop),
                 'n_laptop': len(laptop),
                 'folders': ', '.join(SESSIONS[pid]) if SESSIONS[pid] else '(none found)'})
inventory = pd.DataFrame(rows)
print(inventory.to_string(index=False))

n_folders = int(inventory['n_folders'].sum())
n_units = n_folders * len(SENTENCE_IDS)
print(f"\nTotal recording folders : {n_folders}")
print(f"Total sentence units    : {n_units}  ({n_folders} folders x {len(SENTENCE_IDS)} sentences)")
missing = [label(p) for p in PARTICIPANTS if not SESSIONS[p]]
if missing:
    print(f"WARNING: no data found for {missing} -- check DATA_ROOT.")

# ---------------------------------------------------------------------------
# Which participants can serve as the genuine user?
# ---------------------------------------------------------------------------
# A genuine user needs two domains: training-domain (desktop) sessions and at
# least one cross-domain (laptop) session held out for testing. A participant
# with a single recording folder cannot supply both, so they can only ever act
# as an impostor.
def can_be_genuine(pid):
    laptop = [f for f in SESSIONS[pid] if f in LAPTOP_FOLDERS.get(pid, [])]
    desktop = [f for f in SESSIONS[pid] if f not in laptop]
    return bool(laptop and desktop) or len(desktop) >= 2

ELIGIBLE_GENUINE = [p for p in PARTICIPANTS if SESSIONS[p] and can_be_genuine(p)]
print(f"\nEligible as genuine user : {[label(p) for p in ELIGIBLE_GENUINE]}")
print(f"Impostor-only            : "
      f"{[label(p) for p in PARTICIPANTS if SESSIONS[p] and p not in ELIGIBLE_GENUINE]}")
if len(ELIGIBLE_GENUINE) < 2:
    print("\nNOTE: only one participant has both a training-domain and a")
    print("cross-domain session, so the genuine user cannot be rotated on this")
    print("dataset. Use LOSO_MODE = 'impostor' (rotate the held-out impostor),")
    print("which is exactly the axis the manuscript's Limitations section flags")
    print("as unmeasured. Rotating the genuine user requires collecting a laptop")
    print("session for each remaining participant.")


participant  n_folders  n_desktop  n_laptop                    folders
        P01          2          1         1 202602021946, 202602041210
        P02          1          1         0               202602041444
        P03          1          1         0               202602041345
        P04          1          1         0               202602041705
        P05          1          1         0               202602041413
        P06          1          1         0               202602051540
        P07          1          1         0               202602051645
        P08          1          1         0               202602051727
        P09          1          1         0               202602051800
        P10          1          0         1               202602061716

Total recording folders : 11
Total sentence units    : 33  (11 folders x 3 sentences)

Eligible as genuine user : ['P01']
Impostor-only            : ['P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10']

NOTE

In [121]:
# Part 4: Model and tensor helpers
# ---------------------------------------------------------------------------
# The architecture is unchanged from the proposed model so that results remain
# comparable with the published tables.

class GloveAuthSimpleModel(nn.Module):
    def __init__(self, input_dim=19, hidden_dim=64, num_layers=2, dropout=0.395):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=hidden_dim,
                            num_layers=num_layers, batch_first=True,
                            bidirectional=False,
                            dropout=dropout if num_layers > 1 else 0)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2))

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.lstm(x)
        return self.classifier(h_n[-1])


def make_loader(X, y, batch_size=64, shuffle=False):
    if len(X) == 0:
        return None
    ds = TensorDataset(torch.FloatTensor(X), torch.LongTensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)


def predict(model, loader, device):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(device))
            probs.extend(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            labels.extend(yb.numpy().flatten())
    return np.array(probs), np.array(labels)


In [122]:
# Part 5: Build the train / validation / test partition for one fold
# ---------------------------------------------------------------------------
# Partition contract under STRICT_PROTOCOL:
#
#   train  genuine desktop  (first 80% of each unit)
#          known impostors  (first 80% of each unit)
#   val    genuine desktop  (last 20% of each unit)     <- early stopping only
#          known impostors  (last 20% of each unit)
#   test   genuine laptop   (all)                       <- reported metrics
#          held-out impostor (all)
#
# The test set is therefore never seen during training or model selection.
# In the original notebooks the reported EER was computed on the *same* loader
# used for early stopping, and that loader already contained the held-out
# impostor, so the checkpoint was chosen to minimise loss on the test data.

def _units_for(pid, folders, window_ms, target_len):
    """-> list of (unit_key, feature_array) for one participant."""
    out = []
    for folder in folders:
        for s_id in SENTENCE_IDS:
            drop = True
            if FIRST_REPETITION_SENTENCES is not None:
                drop = s_id in FIRST_REPETITION_SENTENCES
            g_df, k_df = load_unit(DATA_ROOT, pid, folder, s_id,
                                   drop_first_repetition=drop)
            if g_df is None:
                continue
            X = build_windows(g_df, k_df, window_ms, target_len)
            if len(X):
                out.append((f"{pid}/{folder}/{s_id}", X))
    return out


def domain_of(pid, folder):
    """'laptop' if the folder is declared in LAPTOP_FOLDERS, else 'desktop'."""
    return 'laptop' if folder in LAPTOP_FOLDERS.get(pid, []) else 'desktop'


def build_fold(genuine_id, known_impostor_ids, heldout_impostor_id,
               window_ms, target_len, strict=None, verbose=False):
    if strict is None:
        strict = STRICT_PROTOCOL

    laptop = [f for f in SESSIONS[genuine_id] if f in LAPTOP_FOLDERS.get(genuine_id, [])]
    desktop = [f for f in SESSIONS[genuine_id] if f not in laptop]
    if not laptop:
        if len(desktop) < 2:
            raise ValueError(
                f"{label(genuine_id)}: needs either a declared laptop folder or "
                f">=2 recording folders to form a cross-domain test set "
                f"(found {len(SESSIONS[genuine_id])}).")
        laptop = [desktop[-1]]
        desktop = desktop[:-1]
        if verbose:
            print(f"  [{label(genuine_id)}] no laptop folder declared; holding out "
                  f"the latest desktop folder as the test domain.")

    Xtr, ytr, Xva, yva = [], [], [], []
    Xte, yte, gte, dte = [], [], [], []
    units_tr, units_va, units_te = set(), set(), set()

    # --- genuine, training domain -> train + val
    for key, X in _units_for(genuine_id, desktop, window_ms, target_len):
        i_tr, i_va = chronological_split(len(X))
        if len(i_tr): Xtr.append(X[i_tr]); ytr.append(np.ones(len(i_tr), int)); units_tr.add(key)
        if len(i_va): Xva.append(X[i_va]); yva.append(np.ones(len(i_va), int)); units_va.add(key)

    # --- genuine, cross-domain -> test
    for key, X in _units_for(genuine_id, laptop, window_ms, target_len):
        Xte.append(X); yte.append(np.ones(len(X), int)); gte += [key] * len(X)
        dte += [domain_of(genuine_id, key.split('/')[1])] * len(X)
        units_te.add(key)

    # --- known impostors -> train + val
    for imp in known_impostor_ids:
        for key, X in _units_for(imp, SESSIONS[imp], window_ms, target_len):
            i_tr, i_va = chronological_split(len(X))
            if len(i_tr): Xtr.append(X[i_tr]); ytr.append(np.zeros(len(i_tr), int)); units_tr.add(key)
            if len(i_va): Xva.append(X[i_va]); yva.append(np.zeros(len(i_va), int)); units_va.add(key)

    # --- held-out impostor -> test only
    for key, X in _units_for(heldout_impostor_id, SESSIONS[heldout_impostor_id],
                             window_ms, target_len):
        Xte.append(X); yte.append(np.zeros(len(X), int)); gte += [key] * len(X)
        dte += [domain_of(heldout_impostor_id, key.split('/')[1])] * len(X)
        units_te.add(key)

    cat = lambda a, d: (np.concatenate(a) if a else
                        np.empty((0,) + d, dtype=np.float32))
    shape = (target_len, N_CHANNELS)
    Xtr, ytr = cat(Xtr, shape), (np.concatenate(ytr) if ytr else np.empty(0, int))
    Xva, yva = cat(Xva, shape), (np.concatenate(yva) if yva else np.empty(0, int))
    Xte, yte = cat(Xte, shape), (np.concatenate(yte) if yte else np.empty(0, int))
    _g = np.empty(len(gte), dtype=object)
    for _i, _v in enumerate(gte):
        _g[_i] = _v
    gte = _g

    not_legacy = bool(strict)
    if not strict:
        # Legacy: one loader is both the early-stopping set and the reported set.
        Xva = np.concatenate([Xva, Xte]) if len(Xva) else Xte
        yva = np.concatenate([yva, yte]) if len(yva) else yte
        _g = np.empty(len(yva), dtype=object)
        _g[:] = 'legacy'
        Xte, yte, gte = Xva, yva, _g

    fold = dict(genuine=genuine_id, known=list(known_impostor_ids),
                heldout=heldout_impostor_id,
                desktop_folders=desktop, laptop_folders=laptop,
                X_train=Xtr, y_train=ytr, X_val=Xva, y_val=yva,
                X_test=Xte, y_test=yte, g_test=gte,
                d_test=np.array(dte, dtype=object) if not_legacy else None,
                units_train=units_tr, units_val=units_va, units_test=units_te,
                strict=strict)

    if verbose:
        print(f"  train {len(ytr):6d} (genuine {int(ytr.sum()):5d})"
              f" | val {len(yva):6d} (genuine {int(yva.sum()):5d})"
              f" | test {len(yte):6d} (genuine {int(yte.sum()):5d})")

    # Fail loudly and specifically rather than handing back empty arrays that
    # only surface much later as an unrelated error.
    problems = []
    for part, y in (('train', ytr), ('val', yva), ('test', yte)):
        if len(y) == 0:
            problems.append(f"{part} partition is empty")
        elif len(np.unique(y)) < 2 and part != 'val':
            problems.append(f"{part} partition has only class {int(y[0])}")
    if problems:
        raise ValueError(
            f"{label(genuine_id)} vs {label(heldout_impostor_id)}: "
            + "; ".join(problems)
            + ". Run preflight() to see where the windows are being lost "
              "(most often the session_count filter, or a window length that "
              "exceeds the available glove samples).")
    return fold


def preflight(participants=None, window_ms=600, target_len=45, show=True):
    """Per-participant data yield: how many keystrokes survive the repetition
    filter and how many feature windows each recording actually produces.

    Use this whenever a partition comes back empty or smaller than expected.
    discover_sessions() only checks that the CSV files exist; it says nothing
    about how many rows survive filtering, so a folder can be counted in the
    inventory and still contribute zero windows.
    """
    participants = participants or PARTICIPANTS
    rows = []
    for pid in participants:
        for folder in SESSIONS.get(pid, []):
            for s_id in SENTENCE_IDS:
                raw_g, raw_k = load_unit(DATA_ROOT, pid, folder, s_id,
                                         drop_first_repetition=False)
                drop = True
                if FIRST_REPETITION_SENTENCES is not None:
                    drop = s_id in FIRST_REPETITION_SENTENCES
                f_g, f_k = load_unit(DATA_ROOT, pid, folder, s_id,
                                     drop_first_repetition=drop)
                n_raw = len(raw_k) if raw_k is not None else 0
                n_kept = len(f_k) if f_k is not None else 0
                n_win = len(build_windows(f_g, f_k, window_ms, target_len))                         if f_g is not None else 0
                rows.append({'participant': label(pid),
                             'domain': domain_of(pid, folder),
                             'sentence': s_id, 'filtered': drop,
                             'keystrokes_raw': n_raw,
                             'keystrokes_kept': n_kept,
                             'windows': n_win})
    rep = pd.DataFrame(rows)
    if show:
        print(rep.to_string(index=False))
        dead = rep[rep['windows'] == 0]
        print(f"\nTotal windows: {int(rep['windows'].sum())} "
              f"from {len(rep)} sentence units")
        if len(dead):
            print(f"\n{len(dead)} unit(s) produced ZERO windows:")
            print(dead.to_string(index=False))
            lost = dead[(dead['keystrokes_raw'] > 0) & (dead['keystrokes_kept'] == 0)]
            if len(lost):
                print("\n-> These had keystrokes before filtering and none after, so "
                      "the session_count filter removed everything.")
                print("   Set FIRST_REPETITION_SENTENCES = {'1'} (or None-check the "
                      "session_count values in those files).")
    return rep


def build_inner_fold(genuine_id, known_impostor_ids, inner_heldout_id,
                     window_ms, target_len, genuine_holdout_sentence=None,
                     verbose=False):
    """A fold for hyperparameter search only.

    The search must never see the data the paper reports on, so this builder
    stays entirely inside the training domain:

      inner-train  genuine desktop, first 60% of each unit
                   known impostors except `inner_heldout_id`, first 80%
      inner-val    genuine desktop, next 20%           <- early stopping
                   those same impostors, last 20%
      inner-test   genuine desktop, last 20%           <- the tuning objective
                   `inner_heldout_id`, all of it       <- pseudo-unseen impostor

    The genuine cross-domain (laptop) session and the real held-out impostor
    appear nowhere. `inner_heldout_id` is drawn from the KNOWN impostor pool and
    rotates, so the objective is no longer the score against one fixed person --
    which is what the original search optimised for.
    """
    laptop = [f for f in SESSIONS[genuine_id] if f in LAPTOP_FOLDERS.get(genuine_id, [])]
    desktop = [f for f in SESSIONS[genuine_id] if f not in laptop]
    if not desktop:
        raise ValueError(f"{label(genuine_id)} has no training-domain session.")
    if inner_heldout_id in (genuine_id,) or inner_heldout_id not in known_impostor_ids:
        raise ValueError("inner_heldout_id must be one of the KNOWN impostors.")

    Xtr, ytr, Xva, yva, Xte, yte, gte = [], [], [], [], [], [], []
    units_tr, units_va, units_te = set(), set(), set()

    # Genuine side: hold out one whole SENTENCE rather than a later time slice of
    # the same text. A trailing slice of the same recording is near-trivially
    # recognised, which drives the inner EER to exactly 0 for every candidate and
    # leaves the search with nothing to rank. Holding out a sentence keeps the
    # identity task while removing the memorised text.
    if genuine_holdout_sentence is None:
        genuine_holdout_sentence = SENTENCE_IDS[-1]
    for key, X in _units_for(genuine_id, desktop, window_ms, target_len):
        if key.split('/')[2] == genuine_holdout_sentence:
            Xte.append(X); yte.append(np.ones(len(X), int))
            gte += [key] * len(X); units_te.add(key)
            continue
        i_tr, i_va = chronological_split(len(X))
        if len(i_tr): Xtr.append(X[i_tr]); ytr.append(np.ones(len(i_tr), int)); units_tr.add(key)
        if len(i_va): Xva.append(X[i_va]); yva.append(np.ones(len(i_va), int)); units_va.add(key)

    for imp in known_impostor_ids:
        if imp == inner_heldout_id:
            continue
        for key, X in _units_for(imp, SESSIONS[imp], window_ms, target_len):
            i_tr, i_va = chronological_split(len(X))
            if len(i_tr): Xtr.append(X[i_tr]); ytr.append(np.zeros(len(i_tr), int)); units_tr.add(key)
            if len(i_va): Xva.append(X[i_va]); yva.append(np.zeros(len(i_va), int)); units_va.add(key)

    for key, X in _units_for(inner_heldout_id, SESSIONS[inner_heldout_id],
                             window_ms, target_len):
        Xte.append(X); yte.append(np.zeros(len(X), int))
        gte += [key] * len(X); units_te.add(key)

    cat = lambda a: (np.concatenate(a) if a
                     else np.empty((0, target_len, N_CHANNELS), dtype=np.float32))
    ytr = np.concatenate(ytr) if ytr else np.empty(0, int)
    yva = np.concatenate(yva) if yva else np.empty(0, int)
    yte = np.concatenate(yte) if yte else np.empty(0, int)
    _g = np.empty(len(gte), dtype=object)
    for _i, _v in enumerate(gte):
        _g[_i] = _v

    fold = dict(genuine=genuine_id, known=list(known_impostor_ids),
                heldout=inner_heldout_id,
                genuine_holdout_sentence=genuine_holdout_sentence,
                desktop_folders=desktop, laptop_folders=[],
                X_train=cat(Xtr), y_train=ytr, X_val=cat(Xva), y_val=yva,
                X_test=cat(Xte), y_test=yte, g_test=_g, d_test=None,
                units_train=units_tr, units_val=units_va, units_test=units_te,
                strict=True, inner=True)

    for part, y in (('train', ytr), ('val', yva), ('test', yte)):
        if len(y) == 0:
            raise ValueError(f"inner fold {label(genuine_id)} vs "
                             f"{label(inner_heldout_id)}: {part} is empty")
    if len(np.unique(yte)) < 2:
        raise ValueError("inner fold test set has a single class")
    if verbose:
        print(f"    inner: train {len(ytr)} | val {len(yva)} | test {len(yte)} "
              f"(pseudo-unseen impostor {label(inner_heldout_id)}, "
              f"genuine sentence {genuine_holdout_sentence} held out)")
    return fold


def assert_inner_untainted(fold, real_heldout, genuine_id):
    """The search must not have touched the reported test data."""
    every = fold['units_train'] | fold['units_val'] | fold['units_test']
    bad = [u for u in every if u.split('/')[0] == real_heldout]
    assert not bad, f"tuning touched the real held-out impostor: {bad[:3]}"
    lap = set(LAPTOP_FOLDERS.get(genuine_id, []))
    bad = [u for u in every
           if u.split('/')[0] == genuine_id and u.split('/')[1] in lap]
    assert not bad, f"tuning touched the cross-domain session: {bad[:3]}"
    return True


def domain_report(fold):
    """Device composition of the test set, per class.

    This is the quantity that decides whether a low error rate could be an
    artefact of device discrimination rather than identity discrimination. If
    the genuine and impostor halves of the test set come from the same device,
    the model cannot be separating them on device.
    """
    if fold.get('d_test') is None:
        print("  (legacy partition: domain composition not tracked)")
        return None
    y, d = fold['y_test'], fold['d_test']
    rows = []
    for cls, name in [(1, 'genuine'), (0, 'impostor (held-out)')]:
        m = (y == cls)
        doms = sorted(set(d[m].tolist()))
        rows.append({'test class': name, 'n': int(m.sum()),
                     'device(s)': ', '.join(doms) if doms else '-'})
    rep = pd.DataFrame(rows)
    print(rep.to_string(index=False))
    all_doms = set(d.tolist())
    if len(all_doms) == 1:
        print(f"  -> Both test classes come from the {list(all_doms)[0]} domain, "
              f"while training is desktop-only.")
        print("     Device is constant within the test set, so it cannot explain "
              "the separation.")
    else:
        print(f"  -> WARNING: test set mixes devices {sorted(all_doms)}. Check "
              "whether device, rather than identity, is separable here.")
    return rep


def assert_disjoint(fold, deep=False):
    """Verify that no test data reached training or model selection.

    Checked structurally, on recording-unit provenance, which is the actual
    invariant: a (participant, folder, sentence) unit feeds either the training
    domain or the test set, never both. Train and val deliberately DO share unit
    keys -- they are the two halves of one chronological split -- so only the
    test set is required to be unit-disjoint.

    An earlier version compared a lossy hash (the sum of every 37th feature,
    rounded to 6 decimals). At this dataset's size that produces roughly five
    collisions between any two independent partitions, so it reported a leak on
    every fold. Pass deep=True for an exact byte-level confirmation.
    """
    if not fold['strict']:
        return True

    tr, va, te = fold['units_train'], fold['units_val'], fold['units_test']
    if tr & te:
        raise AssertionError(f"LEAK: {len(tr & te)} unit(s) in both train and "
                             f"test, e.g. {sorted(tr & te)[:3]}")
    if va & te:
        raise AssertionError(f"LEAK: {len(va & te)} unit(s) in both val and "
                             f"test, e.g. {sorted(va & te)[:3]}")

    # the held-out impostor must contribute nothing to training or selection
    ho = fold['heldout']
    bad = [u for u in (tr | va) if u.split('/')[0] == ho]
    if bad:
        raise AssertionError(f"LEAK: held-out impostor {label(ho)} appears in "
                             f"train/val via {bad[:3]}")

    # the genuine cross-domain folders must not be in the training domain
    lap = set(fold['laptop_folders'])
    bad = [u for u in (tr | va)
           if u.split('/')[0] == fold['genuine'] and u.split('/')[1] in lap]
    if bad:
        raise AssertionError(f"LEAK: cross-domain folder(s) used for training: "
                             f"{bad[:3]}")

    if deep:
        import hashlib
        dig = lambda X: {hashlib.blake2b(x.tobytes(), digest_size=16).digest()
                         for x in X}
        d_te = dig(fold['X_test'])
        for name, X in (('train', fold['X_train']), ('val', fold['X_val'])):
            hit = d_te & dig(X)
            if hit:
                raise AssertionError(f"LEAK: {len(hit)} identical window(s) "
                                     f"shared between test and {name}")
    return True


In [123]:
# Part 6: Train one model and score the test set
# ---------------------------------------------------------------------------

FIXED_PARAMS = {
    'target_len': 45,
    'hidden_dim': 64,
    'num_layers': 2,
    'lr': 0.0018143021082096381,
    'dropout': 0.39508949789877407,
    'weight_decay': 1e-2,
    'batch_size': 64,
}
PUBLISHED_PARAMS = dict(FIXED_PARAMS)
TUNED_WINDOW_MS = None

# Optuna-Retune.ipynb writes best_params.json. Picking it up here means the
# other notebooks use the retuned configuration without any manual editing.
if os.path.exists("best_params.json"):
    with open("best_params.json", encoding="utf-8") as _f:
        _bp = json.load(_f)
    FIXED_PARAMS = {**FIXED_PARAMS, **_bp['params']}
    TUNED_WINDOW_MS = FIXED_PARAMS.pop('window_ms', None)
    print(f"Loaded best_params.json (objective {_bp['objective'].upper()} = "
          f"{_bp['objective_value']*100:.3f}% at W={_bp['tune_W']}, "
          f"tuned over inner impostors {_bp['inner_heldouts']})")
    if TUNED_WINDOW_MS is not None:
        print(f"  retuned feature window T_w = {TUNED_WINDOW_MS} ms")
else:
    print("best_params.json not found -- using the published hyperparameters.")
EPOCHS = 100
PATIENCE = 8
BATCH_SIZE = 64
# Table 1 of the manuscript reports W = 1..7; go to 10 so the asymptote is
# visible and no future question about a larger window needs a rerun.
SMOOTHING_WINDOWS = list(range(1, 11))

# Persist the per-keystroke test scores of every run. They are what smoothing is
# applied to, so keeping them means any future smoothing window can be evaluated
# without retraining anything. Roughly 4k floats per run: negligible on disk,
# and it removes a whole class of "we only saved W=1,3,5" problem.
SAVE_SCORES = True

# Recurrent stacks trained with a large positive class weight (here
# n_impostor/n_genuine is roughly 7.5) can diverge on an unlucky initialisation:
# gradients explode, the weights saturate, and the network settles on a single
# constant output. Every sample then gets the same score, the ROC collapses to
# the diagonal, and the run reports exactly 50.00% EER at every smoothing
# window -- which is not a measurement but a failed optimisation.
#
# Gradient clipping is the standard remedy and was absent from the original
# notebooks. The checks in train_and_score() catch any collapse that still
# happens, so that it is reported and excluded rather than averaged in.
GRAD_CLIP = 5.0             # max gradient norm; set to None to disable
DEGENERATE_STD = 1e-6       # test-score spread below this counts as collapsed


def train_and_score(fold, seed, params=FIXED_PARAMS,
                    smoothing_windows=SMOOTHING_WINDOWS, quiet=True,
                    epochs=None, patience=None, batch_size=None,
                    on_epoch=None, subsample=None):
    """epochs / patience / batch_size override the module defaults, which the
    hyperparameter search uses to run a cheaper fit per trial.

    on_epoch(epoch, val_loss) is called after every epoch. Raising from it
    aborts the fit, which is how the search prunes a hopeless configuration
    after a few epochs instead of paying for the whole schedule.

    subsample in (0, 1] trains on a stratified fraction of the training windows.
    The search only needs the ranking between configurations, and that survives
    subsampling; the final fits use everything.
    """
    epochs = EPOCHS if epochs is None else epochs
    patience = PATIENCE if patience is None else patience
    batch_size = (params.get('batch_size', BATCH_SIZE) if batch_size is None
                  else batch_size)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    X_tr, y_tr = fold['X_train'], fold['y_train']
    if subsample is not None and 0 < subsample < 1 and len(y_tr):
        rs = np.random.RandomState(seed)
        keep = []
        for cls in np.unique(y_tr):
            idx = np.flatnonzero(y_tr == cls)
            k = max(1, int(round(len(idx) * subsample)))
            keep.append(rs.choice(idx, size=k, replace=False))
        keep = np.sort(np.concatenate(keep))
        X_tr, y_tr = X_tr[keep], y_tr[keep]

    train_loader = make_loader(X_tr, y_tr, batch_size, shuffle=True)
    val_loader   = make_loader(fold['X_val'],   fold['y_val'],   batch_size, shuffle=False)
    test_loader  = make_loader(fold['X_test'],  fold['y_test'],  batch_size, shuffle=False)
    if train_loader is None or val_loader is None or test_loader is None:
        return None

    model = GloveAuthSimpleModel(input_dim=N_CHANNELS,
                                 hidden_dim=params['hidden_dim'],
                                 num_layers=params.get('num_layers', 2),
                                 dropout=params['dropout']).to(DEVICE)

    n_gen = int(y_tr.sum())
    n_imp = int(len(y_tr) - n_gen)
    pos_weight = (n_imp / n_gen) if n_gen > 0 else 1.0
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor([1.0, pos_weight], dtype=torch.float32).to(DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=params['lr'],
                            weight_decay=params.get('weight_decay', 1e-2))

    best_loss, best_w, wait = float('inf'), copy.deepcopy(model.state_dict()), 0
    diverged = False
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE).view(-1)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            if GRAD_CLIP:
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            if not np.isfinite(loss.item()):
                diverged = True
                break
        if diverged:
            break

        model.eval()
        tot = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE).view(-1)
                tot += criterion(model(xb), yb).item() * xb.size(0)
        val_loss = tot / len(fold['y_val'])

        if val_loss < best_loss - 1e-6:
            best_loss, best_w, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break

        if on_epoch is not None:
            on_epoch(epoch, val_loss)      # may raise to abort this fit
    model.load_state_dict(best_w)

    probs, labels = predict(model, test_loader, DEVICE)
    score_std = float(np.std(probs)) if len(probs) else 0.0
    degenerate = (diverged
                  or not np.all(np.isfinite(probs))
                  or score_std < DEGENERATE_STD)

    out = {'epochs_run': epoch + 1, 'val_loss': best_loss,
           'n_test': len(labels), 'n_test_genuine': int(labels.sum()),
           'score_std': score_std, 'diverged': diverged,
           'degenerate': degenerate}
    if SAVE_SCORES:
        out['probs'] = probs
        out['labels'] = labels
        out['groups'] = np.asarray(fold['g_test'], dtype=object)
    for w in smoothing_windows:
        s, l = smooth_grouped(probs, labels, fold['g_test'], w)
        out[w] = eer_and_metrics(l, s) if len(l) else dict(
            eer=np.nan, auc=np.nan, frr_at_far=np.nan, threshold=np.nan)

    if not quiet:
        if degenerate:
            why = ("training loss became non-finite" if diverged else
                   "non-finite scores" if not np.all(np.isfinite(probs)) else
                   f"constant output (score sd {score_std:.2e})")
            print(f"    seed {seed}: COLLAPSED -- {why}. EER is undefined here; "
                  f"the run is excluded, not recorded as 50%. "
                  f"({out['epochs_run']} epochs)")
        else:
            msg = "  ".join(f"W={w}: EER={out[w]['eer']*100:.2f}%"
                            for w in smoothing_windows)
            print(f"    seed {seed}: {msg}  ({out['epochs_run']} epochs, "
                  f"score sd {score_std:.3f})")
    return out


best_params.json not found -- using the published hyperparameters.


In [124]:
# Part 7: Normalization variants
# ---------------------------------------------------------------------------
# build_windows() is redefined here, overriding the copy carried from
# LOSO-CNN-LSTM.ipynb. Everything before the normalization step -- outlier
# clipping, window extraction, the minimum-sample rule, the interpolation to
# target_len -- is byte-identical to the published pipeline. Only the final
# statement differs, and NORM_MODE selects which.

N_RAW_CHANNELS = len(ALL_SENSOR_COLS)          # 19
NORM_MODE = 'zscore'                            # set per arm below

PRESSURE_IDX = [i for i, c in enumerate(ALL_SENSOR_COLS) if c.startswith('R_S_')]
IMU_IDX      = [i for i, c in enumerate(ALL_SENSOR_COLS) if not c.startswith('R_S_')]
PRESSURE_CLIP = 1000.0        # the threshold clip_pressure_outliers() enforces

def out_channels(mode):
    if mode == 'level_feature':   return N_RAW_CHANNELS * 2
    if mode == 'pressure_level':  return N_RAW_CHANNELS + len(PRESSURE_IDX)
    if mode == 'mixed_level':     return N_RAW_CHANNELS + len(PRESSURE_IDX)
    if mode == 'mixed_scale':     return N_RAW_CHANNELS
    return N_RAW_CHANNELS

def build_windows(glove_df, kb_df, window_ms, target_len, min_samples=5):
    g = clip_pressure_outliers(glove_df)
    order = np.argsort(g['glove_timestamp'].values, kind='stable')
    glove_ts = g['glove_timestamp'].values[order]
    glove_vals = g[ALL_SENSOR_COLS].values.astype(np.float32)[order]
    kb_ts = np.sort(kb_df['kb_timestamp_pressed'].values)

    half = window_ms / 2000.0
    out = []
    for ts in kb_ts:
        i0 = np.searchsorted(glove_ts, ts - half, side='left')
        i1 = np.searchsorted(glove_ts, ts + half, side='right')
        seg = glove_vals[i0:i1]
        if len(seg) < min_samples:
            continue
        out.append(_interpolate(seg, target_len))

    if not out:
        return np.empty((0, target_len, out_channels(NORM_MODE)), dtype=np.float32)

    arr = np.asarray(out, dtype=np.float32)
    mean = arr.mean(axis=1, keepdims=True)
    std = arr.std(axis=1, keepdims=True)
    std[std < 1e-6] = 1e-6

    if NORM_MODE == 'zscore':          # published behaviour
        return (arr - mean) / std
    if NORM_MODE == 'scale_only':
        return arr / std
    if NORM_MODE == 'level_feature':
        z = (arr - mean) / std
        level = np.repeat(mean / std, target_len, axis=1)
        return np.concatenate([z, level], axis=2).astype(np.float32)
    if NORM_MODE == 'mixed_scale':
        # Pressure keeps its level as a ratio to the within-window variation;
        # the IMU is Z-scored exactly as published, so its DC components cannot
        # blow up the way they do under scale_only applied to all channels.
        out_arr = (arr - mean) / std
        out_arr[:, :, PRESSURE_IDX] = arr[:, :, PRESSURE_IDX] / std[:, :, PRESSURE_IDX]
        return out_arr.astype(np.float32)
    if NORM_MODE == 'mixed_level':
        # Same information as mixed_scale, presented already separated: the
        # published Z-score for all 19, plus mu/sigma for the pressure channels.
        z = (arr - mean) / std
        ratio = np.repeat((mean / std)[:, :, PRESSURE_IDX], target_len, axis=1)
        return np.concatenate([z, ratio], axis=2).astype(np.float32)
    if NORM_MODE == 'pressure_level':
        # Published Z-score for all 19, plus the pressure level as 10 extra
        # channels on a fixed scale. The IMU representation is untouched.
        z = (arr - mean) / std
        plevel = mean[:, :, PRESSURE_IDX] / PRESSURE_CLIP
        level = np.repeat(plevel, target_len, axis=1)
        return np.concatenate([z, level], axis=2).astype(np.float32)
    raise ValueError(f"unknown NORM_MODE: {NORM_MODE}")


# Channel indices per (modality, normalization). Under level_feature the array
# is [19 z-scored channels | 19 level channels], so a modality takes its slice
# from both halves.
def channel_index(modality, mode):
    press = [i for i, c in enumerate(ALL_SENSOR_COLS) if c.startswith('R_S_')]
    imu   = [i for i, c in enumerate(ALL_SENSOR_COLS) if not c.startswith('R_S_')]
    base = {'pressure': press, 'imu': imu, 'fused': list(range(N_RAW_CHANNELS))}[modality]
    if mode == 'level_feature':
        return base + [i + N_RAW_CHANNELS for i in base]
    if mode in ('pressure_level', 'mixed_level'):
        # The appended block covers the pressure channels only, in the order of
        # PRESSURE_IDX, so the IMU-only arm takes nothing from it and is
        # bit-for-bit the published representation.
        if modality == 'imu':
            return base
        extra = [N_RAW_CHANNELS + k for k, _ in enumerate(PRESSURE_IDX)]
        return base + extra
    return base

def slice_fold(fold, ch_idx):
    out = dict(fold)
    for key in ('X_train', 'X_val', 'X_test'):
        out[key] = np.ascontiguousarray(fold[key][:, :, ch_idx])
    return out

# Self-check on a synthetic window: scale_only must equal z + mean/std, and the
# level half of level_feature must be constant along the time axis.
_a = np.random.RandomState(0).rand(4, 45, 19).astype(np.float32) * 100 + 300
_m, _s = _a.mean(axis=1, keepdims=True), _a.std(axis=1, keepdims=True)
assert np.allclose(_a / _s, (_a - _m) / _s + _m / _s, atol=1e-4)
_lvl = np.repeat(_m / _s, 45, axis=1)
assert np.allclose(_lvl.std(axis=1), 0, atol=1e-5)
# pressure_level must leave the IMU representation identical to the published one
assert channel_index('imu', 'pressure_level') == channel_index('imu', 'zscore')
assert channel_index('imu', 'mixed_level') == channel_index('imu', 'zscore')
assert channel_index('imu', 'mixed_scale') == channel_index('imu', 'zscore')
assert out_channels('pressure_level') == 29
assert out_channels('mixed_level') == 29 and out_channels('mixed_scale') == 19
_NM = 'mixed_scale'
_z = (_a - _m) / _s
_mx = _z.copy(); _mx[:, :, PRESSURE_IDX] = _a[:, :, PRESSURE_IDX] / _s[:, :, PRESSURE_IDX]
assert np.allclose(_mx[:, :, IMU_IDX], _z[:, :, IMU_IDX], atol=1e-6)
assert len(channel_index('fused', 'pressure_level')) == 29
assert len(channel_index('pressure', 'pressure_level')) == 20
print("normalization identities verified")

normalization identities verified


In [125]:
# Part 8: Encoders and the two-branch fusion model
# ---------------------------------------------------------------------------
# Encoder is the published CNN-LSTM with its classification head removed, so a
# single-branch model built from it is identical to GloveAuthSimpleModel.

class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.395):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=hidden_dim,
                            num_layers=num_layers, batch_first=True,
                            bidirectional=False,
                            dropout=dropout if num_layers > 1 else 0)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.lstm(x)
        return h_n[-1]


class TwoBranchFusion(nn.Module):
    """Separate encoders per modality, concatenated before the head.

    The input tensor is ordered [pressure channels | IMU channels]; n_first says
    where the split falls. Because the two encoders share no weights and no
    normalization, the magnitude of one modality cannot suppress the features
    the other learns -- which is the failure mode diagnosed above.
    """

    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.395,
                 n_first=20):
        super().__init__()
        self.n_first = n_first
        self.branch_a = Encoder(n_first, hidden_dim, num_layers, dropout)
        self.branch_b = Encoder(input_dim - n_first, hidden_dim, num_layers, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2))

    def forward(self, x):
        a = self.branch_a(x[:, :, :self.n_first])
        b = self.branch_b(x[:, :, self.n_first:])
        return self.classifier(torch.cat([a, b], dim=-1))


class SingleBranch(nn.Module):
    """Encoder plus the published head; equivalent to GloveAuthSimpleModel."""

    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.395):
        super().__init__()
        self.enc = Encoder(input_dim, hidden_dim, num_layers, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2))

    def forward(self, x):
        return self.classifier(self.enc(x))


def standardize_appended(fold, first_extra):
    """Standardize channels [first_extra:] using TRAINING statistics only.

    Fitting a scaler on the training partition and applying it unchanged to
    validation and test is ordinary practice and not leakage: no test statistic
    enters the transform. It is stated explicitly because the rest of this
    pipeline deliberately avoids any statistic computed across windows.
    """
    out = dict(fold)
    tr = fold['X_train'][:, :, first_extra:]
    mu = tr.mean(axis=(0, 1), keepdims=True)
    sd = tr.std(axis=(0, 1), keepdims=True)
    sd[sd < 1e-6] = 1e-6
    for key in ('X_train', 'X_val', 'X_test'):
        a = fold[key].copy()
        a[:, :, first_extra:] = (a[:, :, first_extra:] - mu) / sd
        out[key] = a
    return out

print("model classes defined")

model classes defined


In [126]:
# Part 6b: Training routine with a selectable model class
# ---------------------------------------------------------------------------
# train_and_score() above, with the model class taken from params. Nothing else
# differs: same early stopping, gradient clipping, collapse detection, smoothing
# and metrics, so any difference between arms is the architecture.

def train_and_score(fold, seed, params,
                    smoothing_windows=SMOOTHING_WINDOWS, quiet=True,
                    epochs=None, patience=None, batch_size=None,
                    on_epoch=None, subsample=None):
    """epochs / patience / batch_size override the module defaults, which the
    hyperparameter search uses to run a cheaper fit per trial.

    on_epoch(epoch, val_loss) is called after every epoch. Raising from it
    aborts the fit, which is how the search prunes a hopeless configuration
    after a few epochs instead of paying for the whole schedule.

    subsample in (0, 1] trains on a stratified fraction of the training windows.
    The search only needs the ranking between configurations, and that survives
    subsampling; the final fits use everything.
    """
    epochs = EPOCHS if epochs is None else epochs
    patience = PATIENCE if patience is None else patience
    batch_size = (params.get('batch_size', BATCH_SIZE) if batch_size is None
                  else batch_size)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    X_tr, y_tr = fold['X_train'], fold['y_train']
    if subsample is not None and 0 < subsample < 1 and len(y_tr):
        rs = np.random.RandomState(seed)
        keep = []
        for cls in np.unique(y_tr):
            idx = np.flatnonzero(y_tr == cls)
            k = max(1, int(round(len(idx) * subsample)))
            keep.append(rs.choice(idx, size=k, replace=False))
        keep = np.sort(np.concatenate(keep))
        X_tr, y_tr = X_tr[keep], y_tr[keep]

    train_loader = make_loader(X_tr, y_tr, batch_size, shuffle=True)
    val_loader   = make_loader(fold['X_val'],   fold['y_val'],   batch_size, shuffle=False)
    test_loader  = make_loader(fold['X_test'],  fold['y_test'],  batch_size, shuffle=False)
    if train_loader is None or val_loader is None or test_loader is None:
        return None

    model = params['model_fn'](input_dim=N_CHANNELS,
                               hidden_dim=params['hidden_dim'],
                               num_layers=params.get('num_layers', 2),
                               dropout=params['dropout']).to(DEVICE)

    n_gen = int(y_tr.sum())
    n_imp = int(len(y_tr) - n_gen)
    pos_weight = (n_imp / n_gen) if n_gen > 0 else 1.0
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor([1.0, pos_weight], dtype=torch.float32).to(DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=params['lr'],
                            weight_decay=params.get('weight_decay', 1e-2))

    best_loss, best_w, wait = float('inf'), copy.deepcopy(model.state_dict()), 0
    diverged = False
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE).view(-1)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            if GRAD_CLIP:
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            if not np.isfinite(loss.item()):
                diverged = True
                break
        if diverged:
            break

        model.eval()
        tot = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE).view(-1)
                tot += criterion(model(xb), yb).item() * xb.size(0)
        val_loss = tot / len(fold['y_val'])

        if val_loss < best_loss - 1e-6:
            best_loss, best_w, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break

        if on_epoch is not None:
            on_epoch(epoch, val_loss)      # may raise to abort this fit
    model.load_state_dict(best_w)

    probs, labels = predict(model, test_loader, DEVICE)
    score_std = float(np.std(probs)) if len(probs) else 0.0
    degenerate = (diverged
                  or not np.all(np.isfinite(probs))
                  or score_std < DEGENERATE_STD)

    out = {'epochs_run': epoch + 1, 'val_loss': best_loss,
           'n_test': len(labels), 'n_test_genuine': int(labels.sum()),
           'score_std': score_std, 'diverged': diverged,
           'degenerate': degenerate}
    if SAVE_SCORES:
        out['probs'] = probs
        out['labels'] = labels
        out['groups'] = np.asarray(fold['g_test'], dtype=object)
    for w in smoothing_windows:
        s, l = smooth_grouped(probs, labels, fold['g_test'], w)
        out[w] = eer_and_metrics(l, s) if len(l) else dict(
            eer=np.nan, auc=np.nan, frr_at_far=np.nan, threshold=np.nan)

    if not quiet:
        if degenerate:
            why = ("training loss became non-finite" if diverged else
                   "non-finite scores" if not np.all(np.isfinite(probs)) else
                   f"constant output (score sd {score_std:.2e})")
            print(f"    seed {seed}: COLLAPSED -- {why}. EER is undefined here; "
                  f"the run is excluded, not recorded as 50%. "
                  f"({out['epochs_run']} epochs)")
        else:
            msg = "  ".join(f"W={w}: EER={out[w]['eer']*100:.2f}%"
                            for w in smoothing_windows)
            print(f"    seed {seed}: {msg}  ({out['epochs_run']} epochs, "
                  f"score sd {score_std:.3f})")
    return out


In [127]:
# Part A: Search configuration
# ---------------------------------------------------------------------------
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

OUTDIR = 'results/optuna-nested'
os.makedirs(OUTDIR, exist_ok=True)

NORM_MODE  = 'mixed_level'
N_CHANNELS = out_channels(NORM_MODE)            # 29
P_IDX      = channel_index('pressure', NORM_MODE)   # 20: 10 z-scored + 10 level
I_IDX      = channel_index('imu', NORM_MODE)        #  9
BOTH_IDX   = P_IDX + I_IDX
N_FIRST    = len(P_IDX)
assert N_FIRST == 20 and len(I_IDX) == 9 and N_CHANNELS == 29

PROPOSED = lambda **kw: TwoBranchFusion(n_first=N_FIRST, **kw)

# T_w is NOT searched here. Section IV-F already settles it with a better
# instrument: ten window lengths, nine rotated folds, three seeds each, under
# the cross-domain protocol, with Holm-corrected Wilcoxon tests -- and it finds
# a wide flat basin rather than an optimum, with the choice inside that basin
# decided by decision latency, T_d = (W-1)*dt_k + T_w/2. This search sees
# neither the cross-domain condition nor the latency cost, so at W = 1 a longer
# window can only look better: it would return the top of the range and mean
# nothing. Leaving T_w fixed keeps the search to the quantities the sweep does
# not address.
WINDOW_MS = 600

# The objective is read WITHOUT smoothing. At W = 3 the desktop-to-desktop
# hold-out is easy enough that trials reach exactly zero error and stop being
# comparable -- the same saturation Section IV-F reports for the
# single-configuration protocol. W = 1 measures the model rather than the
# averaging, and smoothing is applied identically to every configuration
# afterwards, so nothing is lost by tuning without it.
TUNE_W = 1
REPORT_WS = [1, 3, 5]        # recorded per trial; the objective uses TUNE_W

# Ties still happen at W = 1. Rank the worst inner fold rather than the mean --
# the paper's own reading of these numbers is that the spread across impostors
# is what matters -- and break exact ties on the log loss of the objective set,
# which is continuous and cannot saturate. The weight is small enough that it
# only orders configurations the error rate cannot separate.
AGGREGATE = 'worst'          # 'worst' (max over inner folds) or 'mean'
# Set in Part B, once the objective set exists: a tiebreak must be smaller than
# the smallest error rate the objective can resolve, or it stops being a
# tiebreak and starts being the objective. The first run used a flat 1e-3, which
# could reach a tenth of the reported value.
TIEBREAK_WEIGHT = None

# Trial budget. Each trial fits one model per inner fold, so the number of fits
# is N_TRIALS * len(INNER_HELDOUTS) before pruning; the median pruner stops most
# of them within a few epochs.
N_TRIALS       = 200
SEARCH_EPOCHS  = 12       # per trial, against EPOCHS = 100 for a reported fit
SEARCH_PATIENCE = 4
SEARCH_SUBSAMPLE = 0.5    # stratified fraction of the training windows
SEARCH_SEED    = 42

# The search ranks configurations at a fraction of the budget a reported fit
# gets, which favours whatever converges fast, so the ranking is a shortlist and
# not a verdict. No trial is re-fitted at the full budget here, and Part E says
# so rather than leaving it to be assumed.
#
# An earlier draft of this notebook announced a confirmation stage that would
# re-fit the leaders properly. It was dropped deliberately, for reasons Part E
# prints from the trials themselves: the top of the field is a tie the objective
# cannot resolve -- a handful of test windows separates rank 1 from rank 10, and
# the leaders are near-identical configurations -- so re-ranking them at a
# higher budget would reorder noise at the cost of tens of full-length fits. The
# one question that budget confounds, whether the unanimous preference for the
# smallest architecture is real or an artefact of stopping at SEARCH_EPOCHS,
# would not be settled by re-ranking either; Part E reports the preference and
# declines to explain it. Whether this configuration beats the published one is
# a cross-domain question and is settled by the outer rotation, not here.

# The configuration the manuscript reports, enqueued as a trial of its own so
# that Part E can read off where it ranks under this protocol, at this budget,
# without fitting anything further.
PUBLISHED_CONFIG = {'target_len': 45, 'hidden_dim': 64, 'num_layers': 2,
                    'dropout': 0.395, 'lr': 1.814e-3, 'weight_decay': 1e-2}

GENUINE = (DEFAULT_GENUINE if DEFAULT_GENUINE in ELIGIBLE_GENUINE
           else list(ELIGIBLE_GENUINE)[0])
_ok = [p for p in PARTICIPANTS if SESSIONS.get(p)]

# Fixed inner impostors. Three keeps the cost defensible; they are the same
# three the earlier study used, so the two searches are comparable.
INNER_HELDOUTS = [p for p in _ok if p != GENUINE][:3]

print(f"Genuine user      : {label(GENUINE)}")
print(f"Inner impostors   : {[label(p) for p in INNER_HELDOUTS]}")
print(f"T_w               : {WINDOW_MS} ms (fixed; settled by the sweep of Section IV-F)")
print(f"Objective         : {AGGREGATE} EER at W = {TUNE_W} over the inner folds"
      f"  + a log-loss tiebreak sized in Part B")
print(f"Budget            : {N_TRIALS} trials x {len(INNER_HELDOUTS)} folds, "
      f"{SEARCH_EPOCHS} epochs, subsample {SEARCH_SUBSAMPLE}")

Genuine user      : P01
Inner impostors   : ['P02', 'P03', 'P04']
T_w               : 600 ms (fixed; settled by the sweep of Section IV-F)
Objective         : worst EER at W = 1 over the inner folds  + a log-loss tiebreak sized in Part B
Budget            : 200 trials x 3 folds, 12 epochs, subsample 0.5


In [128]:
# Part B: Folds for the search, with the reported test session excluded
# ---------------------------------------------------------------------------
# The genuine user was recorded once on the desktop keyboard and once on the
# laptop, and the laptop session is the test set behind every number in the
# paper. There is no second desktop folder, so the search cannot be given a
# folder-level hold-out without using the session it must not touch.
#
# chronological_split3() exists for exactly this case: it takes a third split
# out of each TRAINING-DOMAIN unit, so the search gets a held-out portion of its
# own and the reported test session is never opened. Each desktop unit is cut
# 60 / 20 / 20 in recording order -- train, early stopping, objective -- and the
# inner impostor contributes the negative class of the objective, as the
# held-out impostor does in the reported protocol.
#
# What this costs. The objective is desktop-to-desktop, while the paper reports
# desktop-to-laptop. A configuration that suits one need not suit the other, and
# Section III-E should say so rather than describe the search as matching the
# reported protocol. What it buys is the thing the section currently admits it
# lacks: an objective computed on data the reported evaluation never sees.

SEARCH_RATIOS = (0.6, 0.2, 0.2)     # train / early stopping / objective

# Every trial rebuilds three folds, and the known-impostor units are shared
# between them, so the same CSVs were being parsed dozens of times. Cache the
# parsed frames -- not the windows, which depend on target_len and would cost
# orders of magnitude more memory.
_LOAD_CACHE = {}

def load_unit_cached(pid, folder, s_id, drop):
    key = (pid, folder, s_id, drop)
    if key not in _LOAD_CACHE:
        _LOAD_CACHE[key] = load_unit(DATA_ROOT, pid, folder, s_id,
                                     drop_first_repetition=drop)
    return _LOAD_CACHE[key]


def units_for(pid, folders, window_ms, target_len):
    """As in Part 5, with the CSV parse cached."""
    out = []
    for folder in folders:
        for s_id in SENTENCE_IDS:
            drop = True
            if FIRST_REPETITION_SENTENCES is not None:
                drop = s_id in FIRST_REPETITION_SENTENCES
            g_df, k_df = load_unit_cached(pid, folder, s_id, drop)
            if g_df is None:
                continue
            X = build_windows(g_df, k_df, window_ms, target_len)
            if len(X):
                out.append((f"{pid}/{folder}/{s_id}", X))
    return out


def genuine_desktop_folders(genuine):
    """The genuine user's training-domain folders; the laptop session is dropped."""
    laptop = LAPTOP_FOLDERS.get(genuine, [])
    return [f for f in SESSIONS[genuine] if f not in laptop]


def search_build_fold(heldout, window_ms, target_len):
    """One inner fold. Same dict contract as build_fold(), never touching the
    genuine user's cross-domain session."""
    desktop = genuine_desktop_folders(GENUINE)
    if not desktop:
        raise ValueError(f"{label(GENUINE)} has no training-domain folder")
    known = [p for p in PARTICIPANTS
             if SESSIONS.get(p) and p not in (GENUINE, heldout)]

    Xtr, ytr, Xva, yva, Xte, yte, gte = [], [], [], [], [], [], []

    def add(dst_X, dst_y, X, cls):
        dst_X.append(X); dst_y.append(np.full(len(X), cls, dtype=int))

    # genuine, training domain -> train / val / objective, in recording order
    for key, X in units_for(GENUINE, desktop, window_ms, target_len):
        i_tr, i_va, i_te = chronological_split3(len(X), SEARCH_RATIOS)
        if len(i_tr): add(Xtr, ytr, X[i_tr], 1)
        if len(i_va): add(Xva, yva, X[i_va], 1)
        if len(i_te):
            add(Xte, yte, X[i_te], 1); gte += [key] * len(i_te)

    # known impostors -> train / val only
    for imp in known:
        for key, X in units_for(imp, SESSIONS[imp], window_ms, target_len):
            i_tr, i_va = chronological_split(len(X))
            if len(i_tr): add(Xtr, ytr, X[i_tr], 0)
            if len(i_va): add(Xva, yva, X[i_va], 0)

    # inner held-out impostor -> objective only
    for key, X in units_for(heldout, SESSIONS[heldout], window_ms, target_len):
        add(Xte, yte, X, 0); gte += [key] * len(X)

    empty = lambda: np.empty((0, target_len, N_CHANNELS), dtype=np.float32)
    cat = lambda a: np.concatenate(a) if a else empty()
    ycat = lambda a: np.concatenate(a) if a else np.empty(0, int)
    g = np.empty(len(gte), dtype=object)
    for i, v in enumerate(gte):
        g[i] = v

    fold = dict(genuine=GENUINE, known=known, heldout=heldout,
                desktop_folders=desktop, laptop_folders=[],
                X_train=cat(Xtr), y_train=ycat(ytr),
                X_val=cat(Xva), y_val=ycat(yva),
                X_test=cat(Xte), y_test=ycat(yte),
                g_test=g, d_test=None, strict=True)

    for part in ('train', 'val', 'test'):
        y = fold[f'y_{part}']
        if len(y) == 0:
            raise ValueError(f"{label(heldout)}: {part} partition is empty")
        if part != 'val' and len(np.unique(y)) < 2:
            raise ValueError(f"{label(heldout)}: {part} carries only class {int(y[0])}")
    return slice_fold(fold, BOTH_IDX)


# Build one fold and print it, so the partition can be checked before 200 trials
# run against it.
_withheld = LAPTOP_FOLDERS.get(GENUINE, [])
_f = search_build_fold(INNER_HELDOUTS[0], WINDOW_MS, 45)
print(f"genuine training folders : {genuine_desktop_folders(GENUINE)}")
print(f"withheld from the search : {_withheld}   <- the reported test session")
print(f"split                    : {SEARCH_RATIOS} of each unit, in order")
print(f"train {len(_f['y_train']):6d} (genuine {int(_f['y_train'].sum()):5d})"
      f" | val {len(_f['y_val']):6d} (genuine {int(_f['y_val'].sum()):5d})"
      f" | objective {len(_f['y_test']):6d} (genuine {int(_f['y_test'].sum()):5d})")
_n = len(_f['y_test'])
TIEBREAK_WEIGHT = 0.1 / _n
print(f"objective resolution     : 1/{_n} = {100.0/_n:.3f}%  "
      f"<- an EER below this is indistinguishable from zero")
print(f"tiebreak weight          : {TIEBREAK_WEIGHT:.2e}  "
      f"(a log loss of 1.0 shifts the objective by a tenth of one step)")
assert _withheld, "no laptop session declared; nothing is being withheld"
del _f


genuine training folders : ['202602021946']
withheld from the search : ['202602041210']   <- the reported test session
split                    : (0.6, 0.2, 0.2) of each unit, in order
train  13845 (genuine  1260) | val   3579 (genuine   420) | objective   2462 (genuine   422)
objective resolution     : 1/2462 = 0.041%  <- an EER below this is indistinguishable from zero
tiebreak weight          : 4.06e-05  (a log loss of 1.0 shifts the objective by a tenth of one step)


In [129]:
# Part C: The objective
# ---------------------------------------------------------------------------
# Reports the running mean EER to the pruner after every epoch of every inner
# fold, so a hopeless configuration is abandoned early rather than paying for
# the whole schedule on all three folds.

SPACE_DOC = """
    T_w          FIXED at 600 ms           settled by the sweep of Section IV-F
    L            25-50, step 5             interpolated sequence length
    d            {32, 64, 128}             LSTM hidden size
    num_layers   {1, 2, 3}                 LSTM depth        (not searched before)
    dropout      0.10-0.50                 dropout rate      (not searched before)
    lr           1e-5 - 5e-3, log-uniform  learning rate
    weight_decay 1e-6 - 1e-3, log-uniform  AdamW weight decay(not searched before)
"""

def suggest(trial):
    return {
        'target_len':   trial.suggest_int('target_len', 25, 50, step=5),
        'hidden_dim':   trial.suggest_categorical('hidden_dim', [32, 64, 128]),
        'num_layers':   trial.suggest_int('num_layers', 1, 3),
        'dropout':      trial.suggest_float('dropout', 0.10, 0.50),
        'lr':           trial.suggest_float('lr', 1e-5, 5e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
    }


def _log_loss(probs, labels, eps=1e-7):
    """Continuous, and never exactly zero, so it can order tied error rates."""
    p = np.clip(np.asarray(probs, dtype=np.float64), eps, 1 - eps)
    y = np.asarray(labels, dtype=np.float64)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def objective(trial):
    p = suggest(trial)
    params = dict(p); params['model_fn'] = PROPOSED
    params['batch_size'] = BATCH_SIZE

    eers, lls = [], []
    per_w = {w: [] for w in REPORT_WS}
    for k, heldout in enumerate(INNER_HELDOUTS):
        try:
            fold = search_build_fold(heldout, WINDOW_MS, p['target_len'])
        except Exception as exc:
            raise optuna.TrialPruned(f"fold {label(heldout)} unbuildable: {exc}")

        # The pruner compares trials at equal step, so step must mean the same
        # thing in every trial. Folds stop early at different epochs, so a
        # running counter does not: step 12 would be fold 0 in one trial and
        # fold 1 in another. Index the fold explicitly instead.
        def on_epoch(epoch, val_loss, _k=k):
            trial.report(float(val_loss), _k * SEARCH_EPOCHS + epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        out = train_and_score(fold, SEARCH_SEED, params,
                              smoothing_windows=REPORT_WS, quiet=True,
                              epochs=SEARCH_EPOCHS, patience=SEARCH_PATIENCE,
                              on_epoch=on_epoch, subsample=SEARCH_SUBSAMPLE)
        if out is None or out['degenerate']:
            raise optuna.TrialPruned(f"collapsed on {label(heldout)}")
        eers.append(out[TUNE_W]['eer'])
        lls.append(_log_loss(out['probs'], out['labels']))
        for w in REPORT_WS:
            per_w[w].append(out[w]['eer'])

    agg = max(eers) if AGGREGATE == 'worst' else float(np.mean(eers))
    value = agg + TIEBREAK_WEIGHT * float(np.mean(lls))

    trial.set_user_attr('eer_per_fold', [float(e) for e in eers])
    trial.set_user_attr('eer_aggregate', float(agg))
    trial.set_user_attr('log_loss', float(np.mean(lls)))
    trial.set_user_attr('n_at_floor', int(sum(e == 0 for e in eers)))
    for w in REPORT_WS:
        trial.set_user_attr(f'eer_W{w}_mean', float(np.mean(per_w[w])))
    return value


print("objective defined; search space:"); print(SPACE_DOC)

objective defined; search space:

    T_w          FIXED at 600 ms           settled by the sweep of Section IV-F
    L            25-50, step 5             interpolated sequence length
    d            {32, 64, 128}             LSTM hidden size
    num_layers   {1, 2, 3}                 LSTM depth        (not searched before)
    dropout      0.10-0.50                 dropout rate      (not searched before)
    lr           1e-5 - 5e-3, log-uniform  learning rate
    weight_decay 1e-6 - 1e-3, log-uniform  AdamW weight decay(not searched before)



In [130]:
# Part D: Run the study
# ---------------------------------------------------------------------------
study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=SEARCH_SEED),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=5))

# Evaluate the manuscript's own configuration under exactly this protocol, so
# Part E can report where it ranks rather than compare it by eye.
study.enqueue_trial(PUBLISHED_CONFIG)
print(f"queued the published configuration as trial 0: {PUBLISHED_CONFIG}")

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

df = study.trials_dataframe()
df.to_csv(f'{OUTDIR}/optuna_trials.csv', index=False)

states = df['state'].value_counts().to_dict()
print(f"\n{len(df)} trials: {states}")

# What the study minimised is not an error rate. It is
#
#     value = AGGREGATE(EER at W = TUNE_W over the inner folds)
#             + TIEBREAK_WEIGHT * mean(log loss)
#
# with AGGREGATE = 'worst', i.e. the MAXIMUM over the inner folds and not their
# mean. Printing `value` as a percentage would present a quantity that carries
# the tiebreak as though it were an EER, and calling it a mean would name the
# wrong statistic. Report the parts separately and let the objective string say
# what was actually optimised. An earlier version of this notebook wrote
# "mean EER at W = 1" into best_params.json; it was neither a mean nor a bare
# EER, and that string is consumed by every downstream notebook.
_bt = study.best_trial
_agg = _bt.user_attrs.get('eer_aggregate')
_ll = _bt.user_attrs.get('log_loss')
_tb = TIEBREAK_WEIGHT * _ll if (_ll is not None) else None
AGG_NAME = {'worst': 'worst-case (max over the inner folds)',
            'mean': 'mean over the inner folds'}[AGGREGATE]
OBJECTIVE_STR = (f"{AGG_NAME} EER at W = {TUNE_W}, "
                 f"plus {TIEBREAK_WEIGHT:.3e} x the mean log loss as a tiebreak")

print(f"\nbest trial  : #{_bt.number}")
print(f"  objective : {study.best_value:.6f}   ({OBJECTIVE_STR})")
if _agg is not None:
    print(f"  of which  : EER {_agg:.6f} ({_agg*100:.3f}%) "
          f"+ tiebreak {_tb:.3e} ({100*_tb/study.best_value:.2f}% of the value)")
print(f"  EER per inner fold at W = {TUNE_W}: "
      + ", ".join(f"{e*100:.3f}%" for e in _bt.user_attrs['eer_per_fold']))
print("best params:")
for k, v in study.best_params.items():
    print(f"   {k:14s} {v}")

best = {
    'params': study.best_params,
    'objective': OBJECTIVE_STR,
    'objective_value': study.best_value,
    # The three pieces of `objective_value`, so no reader has to reverse it.
    'objective_components': {
        'aggregate': AGGREGATE,
        'eer_aggregate': _agg,
        'log_loss': _ll,
        'tiebreak_weight': TIEBREAK_WEIGHT,
        'tiebreak_contribution': _tb,
    },
    'eer_per_fold': _bt.user_attrs.get('eer_per_fold'),
    'tune_W': TUNE_W,
    'inner_heldouts': [label(p) for p in INNER_HELDOUTS],
    'window_ms_fixed': WINDOW_MS,
    'n_trials': len(df),
    'n_pruned': int(states.get('PRUNED', 0)),
    'n_complete': int(states.get('COMPLETE', 0)),
    'architecture': 'TwoBranchFusion (20 pressure + 9 inertial channels)',
    'search_split': list(SEARCH_RATIOS),
    'reported_session_withheld': LAPTOP_FOLDERS.get(GENUINE, []),
    # Provenance of the ranking. Downstream notebooks consume this file, so the
    # budget it was chosen at travels with it instead of living only in a
    # comment here.
    'selection': {
        'stage': 'search',
        'best_trial': _bt.number,
        'search_epochs': SEARCH_EPOCHS,
        'search_patience': SEARCH_PATIENCE,
        'search_subsample': SEARCH_SUBSAMPLE,
        'search_seed': SEARCH_SEED,
        'seeds_per_fold': 1,
        'refitted_at_full_budget': False,
        'note': (f'ranked at {SEARCH_EPOCHS} epochs, subsample '
                 f'{SEARCH_SUBSAMPLE}, one seed per inner fold, against '
                 f'{EPOCHS} epochs on all windows for a reported run. No trial '
                 'was re-fitted at the full budget: see Part E for why, and for '
                 'what that leaves unresolved.'),
    },
    'independence': ('the objective is a chronological tail of the genuine '
                     "user's desktop recording; the cross-domain session the "
                     'paper reports on is never opened. The inner impostors '
                     'also appear as outer folds in Table 1.'),
}
with open(f'{OUTDIR}/best_params.json', 'w', encoding='utf-8') as f:
    json.dump(best, f, indent=2)
print(f"\nwritten to {OUTDIR}/best_params.json")


[I 2026-08-13 11:33:35,940] A new study created in memory with name: no-name-1d347650-ee69-40f6-afc2-5f79fd0df0a4


queued the published configuration as trial 0: {'target_len': 45, 'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.395, 'lr': 0.001814, 'weight_decay': 0.01}


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-08-13 11:34:45,277] Trial 0 finished with value: 0.009194848014854293 and parameters: {'target_len': 45, 'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.395, 'lr': 0.001814, 'weight_decay': 0.01}. Best is trial 0 with value: 0.009194848014854293.
[I 2026-08-13 11:35:33,521] Trial 1 finished with value: 0.15193440401607178 and parameters: {'target_len': 35, 'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.16239780813448107, 'lr': 1.4347159517201392e-05, 'weight_decay': 0.0003967605077052988}. Best is trial 0 with value: 0.009194848014854293.
[I 2026-08-13 11:37:27,594] Trial 2 finished with value: 0.009848932817143401 and parameters: {'target_len': 40, 'hidden_dim': 128, 'num_layers': 3, 'dropout': 0.18493564427131048, 'lr': 3.095566460242367e-05, 'weight_decay': 3.549878832196506e-06}. Best is trial 0 with value: 0.009194848014854293.
[I 2026-08-13 11:38:24,496] Trial 3 finished with value: 0.006039873940003924 and parameters: {'target_len': 30, 'hidden_dim': 32, 'num_layers': 2

In [136]:
# Part E: What the search actually established
# ---------------------------------------------------------------------------
# Read-only. This cell fits nothing: every number below comes from the trials
# the search already ran, and it can be re-run from optuna_trials.csv alone.
#
# The manuscript's own configuration was enqueued as the first trial, so it was
# evaluated under exactly this protocol, at exactly this budget, alongside every
# other configuration. Its rank is therefore a direct read-off and needs no
# re-fitting; an earlier plan to confirm a shortlist at the full budget was
# dropped because it would not have answered anything this cell cannot, at the
# price of tens of full-length fits. What that decision leaves unresolved is
# stated at the end rather than left implicit.

_trials_csv = f'{OUTDIR}/optuna_trials.csv'
if 'study' in globals():
    _d = study.trials_dataframe()
elif os.path.exists(_trials_csv):
    _d = pd.read_csv(_trials_csv)
else:
    raise FileNotFoundError(f"no study in scope and no {_trials_csv}")

field = _d[_d['state'] == 'COMPLETE'].sort_values('value').reset_index(drop=True)
field['rank'] = field.index + 1
PCOLS = [c for c in field.columns if c.startswith('params_')]
pname = lambda c: c[len('params_'):]

# Identify the published configuration by its parameters rather than by trial
# number: enqueue_trial happens to make it trial 0, but that is a convention.
def _is_published(row):
    for k, v in PUBLISHED_CONFIG.items():
        col = f'params_{k}'
        if col not in row:
            return False
        a, b = row[col], v
        if isinstance(b, float):
            if abs(float(a) - b) > 1e-3 * max(1.0, abs(b)):
                return False
        elif a != b:
            return False
    return True

_pub = field[field.apply(_is_published, axis=1)]
n = len(field)

print("=== The published configuration, scored as a trial of this search ===")
if _pub.empty:
    print("  the published configuration is not among the completed trials "
          "(it may have been pruned); nothing can be read off.")
else:
    r = _pub.iloc[0]
    b = field.iloc[0]
    print(f"  published : rank {int(r['rank'])} of {n}   "
          f"{AGGREGATE}-fold EER {r['user_attrs_eer_aggregate']*100:.3f}%")
    print(f"  selected  : rank 1          "
          f"{AGGREGATE}-fold EER {b['user_attrs_eer_aggregate']*100:.3f}%   "
          f"(trial {int(b['number'])})")
    for w in REPORT_WS:
        col = f'user_attrs_eer_W{w}_mean'
        if col in field.columns:
            print(f"    mean EER at W = {w}: published "
                  f"{r[col]*100:.4f}%   selected {b[col]*100:.4f}%")
    print(f"  -> {int(r['rank']) - 1} of {n} completed configurations score better "
          f"than the published one on this objective.")
    print("     This is an INNER-fold result: desktop-to-desktop, three impostors,")
    print("     no cross-domain condition. Section III-E of the manuscript reports")
    print("     that the inner folds once ranked two configurations in the opposite")
    print("     order to the outer evaluation, so this gap does not predict the")
    print("     reported EER and must not be quoted as an improvement to it. The")
    print("     outer rotation is the only thing that settles that.")

# --- how sharp is the winner? -----------------------------------------------
# The objective cannot resolve better than one test window. Express the spread
# across the top of the field in those units, so "best" is not read as a
# meaningful margin when it is a handful of misclassified windows.
RESOLUTION = TIEBREAK_WEIGHT * 10 * 1.0        # = 1/n_objective, by construction
_n_obj = int(round(1.0 / RESOLUTION))
print(f"\n=== How sharp is the optimum? (objective resolution 1/{_n_obj} = "
      f"{RESOLUTION*100:.3f}%) ===")
for k in (1, 3, 5, 10, 20):
    if k <= n:
        e = field.iloc[k - 1]['user_attrs_eer_aggregate']
        print(f"  rank {k:3d}: {e*100:.3f}%   = {e*_n_obj:5.1f} test windows")
_tied = int((field['user_attrs_eer_aggregate']
             <= field.iloc[0]['user_attrs_eer_aggregate'] + RESOLUTION).sum())
print(f"  {_tied} trials sit within one resolution step of the best, so the top of")
print(f"  the field is a tie the objective cannot break. The configuration this")
print(f"  search returns should be described as one drawn from that tie, not as")
print(f"  the optimum of the space.")

# --- what does the top of the field have in common? --------------------------
# If the leaders are unanimous on an architectural choice, say so: it is either
# a real finding about this corpus or an artefact of the reduced budget, and
# this search cannot tell which.
TOP_K = min(10, n)
print(f"\n=== What the top {TOP_K} agree on ===")
_flag = []
for c in PCOLS:
    k = pname(c)
    top, all_ = field[c].head(TOP_K), field[c]
    if pd.api.types.is_float_dtype(all_) and all_.nunique() > 8:
        print(f"  {k:14s} top {TOP_K}: {top.min():.4g} to {top.max():.4g}"
              f"   (field {all_.min():.4g} to {all_.max():.4g})")
    else:
        vc = top.value_counts()
        share = vc.iloc[0] / TOP_K
        print(f"  {k:14s} top {TOP_K}: " +
              ", ".join(f"{v}x{int(cnt)}" for v, cnt in vc.items()) +
              f"   (field: " + ", ".join(f"{v}x{int(cnt)}"
                                         for v, cnt in all_.value_counts().items()) + ")")
        if share == 1.0 and all_.nunique() > 1:
            _flag.append((k, vc.index[0]))
if _flag:
    print("\n  unanimous among the leaders: " +
          ", ".join(f"{k} = {v}" for k, v in _flag))
    print(f"  Trials were ranked at {SEARCH_EPOCHS} epochs on a "
          f"{SEARCH_SUBSAMPLE} subsample with one seed, a budget that favours")
    print( "  whatever converges fastest. A unanimous preference for the smallest,")
    print( "  fastest members of the space is therefore consistent with two")
    print( "  explanations -- that they are genuinely better on this corpus, or")
    print( "  that the budget selected them -- and this search cannot separate the")
    print( "  two. Report the preference; do not report a mechanism for it.")

print("\n=== What this search does and does not establish ===")
print( "  DOES : a configuration chosen without opening the cross-domain session")
print( "         the paper reports on. That is the flaw Section III-E admits.")
print(f"  DOES NOT: an optimum. T_w was fixed at {WINDOW_MS} ms and not searched;")
print(f"         trials were ranked at {SEARCH_EPOCHS} epochs, subsample "
      f"{SEARCH_SUBSAMPLE}, one seed,")
print( "         and none was re-fitted at the budget a reported run gets; the top")
print( "         of the field is within the resolution of the objective; and the")
print(f"         inner impostors {[label(p) for p in INNER_HELDOUTS]} also appear as outer folds")
print( "         in Table 1, so the impostor set is not independent of Table 1.")
print( "  The question of whether this configuration is actually better than the")
print( "  published one is settled by the outer cross-domain rotation, not here.")


=== The published configuration, scored as a trial of this search ===
  published : rank 101 of 115   worst-fold EER 0.919%
  selected  : rank 1          worst-fold EER 0.270%   (trial 172)
    mean EER at W = 1: published 0.5736%   selected 0.2372%
    mean EER at W = 3: published 0.0263%   selected 0.0085%
    mean EER at W = 5: published 0.0000%   selected 0.0000%
  -> 100 of 115 completed configurations score better than the published one on this objective.
     This is an INNER-fold result: desktop-to-desktop, three impostors,
     no cross-domain condition. Section III-E of the manuscript reports
     that the inner folds once ranked two configurations in the opposite
     order to the outer evaluation, so this gap does not predict the
     reported EER and must not be quoted as an improvement to it. The
     outer rotation is the only thing that settles that.

=== How sharp is the optimum? (objective resolution 1/2462 = 0.041%) ===
  rank   1: 0.270%   =   6.7 test windows
  ran